In [1]:
import duckdb
from understatapi import UnderstatClient
from matplotlib import pyplot as plt
import pandas as pd

In [ ]:
# Example: liverpool current player valuations over time

q = """
SELECT P.player_id, P.name, V.date, V.market_value_in_eur
FROM read_csv_auto('https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/players.csv.gz') P
JOIN read_csv_auto('https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/player_valuations.csv.gz') V ON P.player_id = V.player_id
WHERE current_club_domestic_competition_id = 'GB1' AND last_season = 2025
AND P.current_club_id = 31
ORDER BY P.player_id, V.date
"""

duckdb.sql(q).show()

In [ ]:
q = """
SELECT *
FROM read_csv_auto('https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/appearances.csv.gz') A
LIMIT 10
"""

duckdb.sql(q).show()

In [2]:
# Issues with getting FBRef data without avoiding 403 errors so we use understat here
understat = UnderstatClient()

# Get all player data for a league/season
epl_players = understat.league(league="EPL").get_player_data(season="2025")

In [3]:
player_df = pd.DataFrame(epl_players)

In [4]:
player_df["position"].unique()

<StringArray>
[    'F S',     'F M',   'F M S',     'M S',       'F',       'M',   'D M S',
     'D S',     'D M',       'D', 'D F M S',       'S',      'GK',    'GK S']
Length: 14, dtype: str

In [5]:
'D F M S'.split(" ")

['D', 'F', 'M', 'S']

In [6]:
epl_players[:3]

[{'id': '8260',
  'player_name': 'Erling Haaland',
  'games': '27',
  'time': '2259',
  'goals': '22',
  'xG': '22.47195105999708',
  'assists': '7',
  'xA': '4.683228434994817',
  'shots': '96',
  'key_passes': '20',
  'yellow_cards': '1',
  'red_cards': '0',
  'position': 'F S',
  'team_title': 'Manchester City',
  'npg': '19',
  'npxG': '19.427275620400906',
  'xGChain': '24.82650714367628',
  'xGBuildup': '3.465328110381961'},
 {'id': '13222',
  'player_name': 'Thiago',
  'games': '27',
  'time': '2302',
  'goals': '17',
  'xG': '18.74260649085045',
  'assists': '1',
  'xA': '2.971277406439185',
  'shots': '63',
  'key_passes': '15',
  'yellow_cards': '5',
  'red_cards': '0',
  'position': 'F S',
  'team_title': 'Brentford',
  'npg': '11',
  'npxG': '13.414424546062946',
  'xGChain': '16.38330163434148',
  'xGBuildup': '2.8900308422744274'},
 {'id': '11363',
  'player_name': 'Antoine Semenyo',
  'games': '26',
  'time': '2340',
  'goals': '13',
  'xG': '9.839814426377416',
  'assis

In [7]:
# Find player info
list(filter(lambda p: p['player_name'] == 'Mohamed Salah', epl_players))

[{'id': '1250',
  'player_name': 'Mohamed Salah',
  'games': '19',
  'time': '1626',
  'goals': '4',
  'xG': '6.514106601476669',
  'assists': '6',
  'xA': '5.435673952102661',
  'shots': '48',
  'key_passes': '42',
  'yellow_cards': '1',
  'red_cards': '0',
  'position': 'F M S',
  'team_title': 'Liverpool',
  'npg': '3',
  'npxG': '5.752937763929367',
  'xGChain': '11.591482192277908',
  'xGBuildup': '3.587615504860878'}]

In [8]:
# Get all player matches
player_matches = understat.player(player='13835').get_match_data()

In [9]:
# Convert to dataframe
player_matches_df = pd.DataFrame(player_matches)

# Sort by date
player_matches_df = player_matches_df.sort_values(by="date")

player_matches_df.head()

,goals,shots,xG,time,position,h_team,a_team,h_goals,a_goals,date,id,season,roster_id,xA,assists,key_passes,npg,npxG,xGChain,xGBuildup
10,0,3,0.5239394903182983,78,FW,Leeds,Everton,1,0,2025-08-18,28787,2025,732652,0,0,0,0,0.5239394903182983,0.5561109185218811,0.03217145800590515
9,0,0,0,61,FW,Arsenal,Leeds,5,0,2025-08-23,28793,2025,733410,0,0,0,0,0,0.024242989718914032,0.024242989718914032
8,0,1,0.058323636651039124,7,Sub,Leeds,Tottenham,1,2,2025-10-04,28844,2025,742135,0,0,0,0,0.058323636651039124,0.058323636651039124,0
7,0,2,0.14811818301677704,16,Sub,Burnley,Leeds,2,0,2025-10-18,28849,2025,744156,0,0,0,0,0.14811818301677704,0.16506358981132507,0.016945406794548035
6,0,0,0,12,Sub,Nottingham Forest,Leeds,3,1,2025-11-09,28884,2025,750869,0,0,0,0,0,0.021056337282061577,0.021056337282061577


In [ ]:
# Rolling stats over games

stats = ["goals", "xG", "assists", "xA", "key_passes", "xGChain", "xGBuildup"]
rolling_stats = list(map(lambda s: f"rolling_{s}_per_90", stats))

window_size = 10

# Get minute counts first
player_matches_df["rolling_min"] = player_matches_df["time"].rolling(window_size).sum()

# Rolling stats/90 for each
for stat, rolling_stat_name in zip(stats, rolling_stats):

    rolling_stat = player_matches_df[stat].rolling(window_size).sum()

    player_matches_df[rolling_stat_name] = rolling_stat / player_matches_df["rolling_min"] * 90

# Only date plus stats
stats_df = player_matches_df[["date"] + rolling_stats]

# drop nas (first window_size - 1)
stats_df = stats_df.dropna()

stats_df.head(10)

In [ ]:
plt.plot(stats_df["date"], stats_df["rolling_goals_per_90"], color="red", label="rolling_goals_per_90")
plt.plot(stats_df["date"], stats_df["rolling_assists_per_90"], color="blue", label="rolling_assists_per_90")
plt.legend()
plt.xlabel("Date")
plt.xticks(stats_df["date"][::80])

In [ ]:
# Use helpers

from preprocess import *
understat = UnderstatClient()

In [ ]:
# Look at forwards

f_ids = get_player_ids(understat, {"F"})

In [ ]:
# Get each forward's stats df

stats = ["goals", "xG", "assists", "xA", "key_passes", "xGChain", "xGBuildup"]
window_size = 10

for f_id in f_ids[:3]:
    f_stats = get_player_stats_df(understat, f_id, stats, window_size)
    print(f_stats.head())
    